In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_REV = '1cdfd44'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', '-q', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '-q', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))


In [ ]:
from spider.exp4_data import find_exp4_checkpoint, find_exp4_data

assert shutil.which('zstd'), 'zstd is required for a single-file transfer artifact'
prepared = find_exp4_data('/kaggle/input')
checkpoint = find_exp4_checkpoint('/kaggle/input', 250)
training_output = checkpoint.parents[1]
target = Path('/kaggle/working/cloud-transfer')
target.mkdir(parents=True, exist_ok=True)
artifacts = [
    (prepared, target / 'prepared-data.tar.zst'),
    (training_output, target / 'step_0250.tar.zst'),
]
for source, archive in artifacts:
    print({'event': 'archive_start', 'source': str(source), 'archive': str(archive)}, flush=True)
    subprocess.run([
        'tar', '--use-compress-program=zstd -3 -T0', '-cf', str(archive),
        '-C', str(source.parent), source.name,
    ], check=True)
    print({'event': 'archive_complete', 'archive': str(archive), 'bytes': archive.stat().st_size}, flush=True)
summary = {path.name: path.stat().st_size for _, path in artifacts}
(target / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print({'event': 'cloud_transfer_complete', 'summary': summary}, flush=True)
